In [1]:
import biom
import pandas as pd
import subprocess
import os
from collections import defaultdict

# 1. Caricamento della tabella
print("Caricamento della tabella BIOM...")
table = biom.load_table('emp_deblur_150bp.release1.biom')
all_sample_ids = list(table.ids(axis='sample'))

batch_size = 100
batches = [all_sample_ids[i:i+batch_size] for i in range(0, len(all_sample_ids), batch_size)]
print(f'Totale batch: {len(batches)} da {batch_size} campioni ciascuno')

results = []
# Usiamo un dizionario di set per accumulare le OTU per ogni funzione senza duplicati
global_otu2func = defaultdict(set)

# 2. Processamento a batch
for idx, batch_ids in enumerate(batches):
    print(f'Batch {idx+1}/{len(batches)}...', end=' ', flush=True)
    
    # Filtra ai campioni del batch e rimuovi le OTU con somma 0
    table_batch = table.filter(batch_ids, axis='sample', inplace=False)
    table_batch = table_batch.filter(
        lambda val, id_, md: val.sum() > 0,
        axis='observation', inplace=False
    )
    
    # Salva batch BIOM
    biom_batch = f'batch_{idx}.biom'
    with biom.util.biom_open(biom_batch, 'w') as f:
        table_batch.to_hdf5(f, f'batch_{idx}')
    
    # Lancia FAPROTAX sul batch
    out_batch = f'functional_batch_{idx}.tsv'
    otu2func_batch = f'otu2func_batch_{idx}.tsv'
    cmd = [
        'python', 'collapse_table.py',
        '-i', biom_batch,
        '-o', out_batch,
        '-g', 'FAPROTAX.txt',
        '--collapse_by_metadata', 'taxonomy',
        '--out_groups2records_table_dense', otu2func_batch,
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0 and os.path.exists(out_batch):
        df_batch = pd.read_csv(out_batch, sep='\t', index_col=0)
        results.append(df_batch)
        print(f'OK ({df_batch.shape})', end=' ')
    else:
        print(f'ERRORE: {result.stderr}')
    
    # Raccogli e unisci la mappa OTU -> funzioni riga per riga
    if os.path.exists(otu2func_batch):
        with open(otu2func_batch, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) > 1: # Se c'è almeno la funzione e una OTU
                    func_group = parts[0]
                    otus = parts[1:]
                    # Aggiunge le OTU al set (i set ignorano automaticamente i duplicati)
                    global_otu2func[func_group].update(otus)
        print('[otu2func OK]')
    else:
        print('[otu2func MANCANTE]')
    
    # Rimuovi file temporanei 
    os.remove(biom_batch)
    if os.path.exists(out_batch): os.remove(out_batch)
    if os.path.exists(otu2func_batch): os.remove(otu2func_batch)

# 3. Unione dei risultati
if results:
    print('\nUnisco i batch (tabella funzionale)...')
    df_functional = pd.concat(results, axis=1).fillna(0)
    print('Shape finale:', df_functional.shape)
    df_functional.to_csv('functional_table_final.tsv', sep='\t')
    print('Salvato functional_table_final.tsv!')
else:
    print('\nNessun batch processato con successo.')

if global_otu2func:
    print('\nSalvataggio mappa OTU->funzioni (otu2func_final.tsv)...')
    with open('otu2func_final.tsv', 'w') as out_f:
        # Scriviamo il file nello stesso formato di FAPROTAX
        for func_group, otus in sorted(global_otu2func.items()):
            # Uniamo il nome della funzione e tutte le sue OTU con tabulazioni
            line_to_write = f"{func_group}\t" + "\t".join(sorted(otus)) + "\n"
            out_f.write(line_to_write)
    print(f'Salvato otu2func_final.tsv con {len(global_otu2func)} funzioni uniche!')

print('\nElaborazione completata.')

Caricamento della tabella BIOM...
Totale batch: 175 da 100 campioni ciascuno
Batch 1/175... OK ((92, 100)) [otu2func OK]
Batch 2/175... OK ((92, 100)) [otu2func OK]
Batch 3/175... OK ((92, 100)) [otu2func OK]
Batch 4/175... OK ((92, 100)) [otu2func OK]
Batch 5/175... OK ((92, 100)) [otu2func OK]
Batch 6/175... OK ((92, 100)) [otu2func OK]
Batch 7/175... OK ((92, 100)) [otu2func OK]
Batch 8/175... OK ((92, 100)) [otu2func OK]
Batch 9/175... OK ((92, 100)) [otu2func OK]
Batch 10/175... OK ((92, 100)) [otu2func OK]
Batch 11/175... OK ((92, 100)) [otu2func OK]
Batch 12/175... OK ((92, 100)) [otu2func OK]
Batch 13/175... OK ((92, 100)) [otu2func OK]
Batch 14/175... OK ((92, 100)) [otu2func OK]
Batch 15/175... OK ((92, 100)) [otu2func OK]
Batch 16/175... OK ((92, 100)) [otu2func OK]
Batch 17/175... OK ((92, 100)) [otu2func OK]
Batch 18/175... OK ((92, 100)) [otu2func OK]
Batch 19/175... OK ((92, 100)) [otu2func OK]
Batch 20/175... OK ((92, 100)) [otu2func OK]
Batch 21/175... OK ((92, 100)) [

In [2]:
from biom.table import Table
import numpy as np

# df_functional ha righe = funzioni, colonne = campioni
# Assicurati che sia nel formato giusto (funzioni × campioni)
data = df_functional.values.astype(float)
obs_ids = list(df_functional.index)    # funzioni FAPROTAX
samp_ids = list(df_functional.columns) # campioni

# Crea la BIOM table
functional_biom = Table(
    data=data,
    observation_ids=obs_ids,
    sample_ids=samp_ids
)

print('Shape BIOM funzionale:', functional_biom.shape)

# Salva
with biom.util.biom_open('functional_table_final.biom', 'w') as f:
    functional_biom.to_hdf5(f, 'faprotax_output')

print('Salvato functional_table_final.biom!')

# Verifica
table_check = biom.load_table('functional_table_final.biom')
print('Verifica shape:', table_check.shape)
print('Prime funzioni:', list(table_check.ids(axis='observation'))[:5])

Shape BIOM funzionale: (92, 17483)
Salvato functional_table_final.biom!
Verifica shape: (92, 17483)
Prime funzioni: ['methanotrophy', 'acetoclastic_methanogenesis', 'methanogenesis_by_disproportionation_of_methyl_groups', 'methanogenesis_using_formate', 'methanogenesis_by_CO2_reduction_with_H2']
